In [ ]:
# ============================================================
#  FINAL PRODUCTION SCRIPT — 528 Hz + NATURAL FEMALE VOICE (gTTS)
#  WAV + PDF — Google Colab Ready
# ============================================================

# System + Python deps
!pip install -q pydub reportlab matplotlib scipy requests gTTS

import numpy as np
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as wav_write
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from datetime import datetime
from pydub import AudioSegment
import subprocess, io, requests
from IPython.display import Audio, display, IFrame
from gtts import gTTS  # <-- NEW (no API required)

# Detect Colab for auto-download
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


# ============================================================
# USER SETTINGS
# ============================================================
DURATION_SEC = 600          # total audio length in seconds
SAMPLE_RATE = 44100         # Hz
MASTER_VOLUME = 0.25        # final normalized peak
APPLY_FADE = True           # quick fade in/out on tone

CENTER_FREQ_HZ = 528        # main Solfeggio frequency
AFFIRMATION_TEXT = "You're beautiful!"
AFFIRMATION_INTERVAL = 30.0  # seconds between affirmations

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
WAV_FILENAME = f"528Hz_HealingSolfeggio_{STAMP}.wav"
PDF_FILENAME = f"528Hz_HealingSolfeggio_Info_{STAMP}.pdf"


# ============================================================
# HELPERS
# ============================================================
def sine_wave(freq_hz, duration, sr):
    t = np.linspace(0.0, duration, int(sr * duration), endpoint=False)
    return np.sin(2.0 * np.pi * freq_hz * t)

def apply_fade_io(wave, sr, fade_ms=20):
    n = len(wave)
    k = int(sr * (fade_ms / 1000.0))
    if k < 1 or 2 * k > n:
        return wave
    fade_in = np.linspace(0.0, 1.0, k)
    fade_out = fade_in[::-1]
    wave[:k] *= fade_in
    wave[-k:] *= fade_out
    return wave

def get_latex_image(latex_code):
    url = f"https://latex.codecogs.com/png.image?\\dpi{{150}} {latex_code}"
    r = requests.get(url)
    r.raise_for_status()
    return ImageReader(io.BytesIO(r.content))


# ============================================================
# 1. MAIN 528 Hz TONE
# ============================================================
tone = sine_wave(CENTER_FREQ_HZ, DURATION_SEC, SAMPLE_RATE).astype(np.float32)
if APPLY_FADE:
    tone = apply_fade_io(tone, SAMPLE_RATE)

audio = tone.copy()  # mono track


# ============================================================
# 2. NATURAL FEMALE VOICE — gTTS (NO API REQUIRED)
# ============================================================
def generate_affirmation_wav(text, outfile):
    """
    Generate a natural female voice using gTTS (Google Translate TTS).
    No API key needed. Produces a real human-sounding female voice.
    """
    tts = gTTS(text=text, lang="en", slow=False)
    tts.save(outfile)

generate_affirmation_wav(AFFIRMATION_TEXT, "affirmation.wav")

# Load into numpy for mixing (unchanged)
speech_segment = AudioSegment.from_mp3("affirmation.wav").set_frame_rate(SAMPLE_RATE).set_channels(1)
speech_samples = np.array(speech_segment.get_array_of_samples()).astype(np.float32)
speech_samples /= (np.max(np.abs(speech_samples)) + 1e-9)

speech_track = np.zeros_like(audio)
pos = 0.0
while pos < DURATION_SEC:
    start_idx = int(pos * SAMPLE_RATE)
    slice_len = min(len(speech_samples), len(speech_track) - start_idx)
    if slice_len > 0:
        speech_track[start_idx:start_idx + slice_len] = speech_samples[:slice_len]
    pos += AFFIRMATION_INTERVAL

speech_volume = 0.40
audio = audio + speech_volume * speech_track

peak = float(np.max(np.abs(audio)))
if peak > 0:
    audio = (audio / peak) * MASTER_VOLUME


# ============================================================
# 3. EXPORT WAV + INLINE PLAYBACK + AUTO-DOWNLOAD
# ============================================================
wav_int16 = (audio * 32767.0).astype(np.int16)
wav_write(WAV_FILENAME, SAMPLE_RATE, wav_int16)

print("Exported WAV:", WAV_FILENAME)
display(Audio(WAV_FILENAME))

if IN_COLAB:
    files.download(WAV_FILENAME)


# ============================================================
# 4. PDF GENERATOR
# ============================================================
def generate_pdf(filename, freq, amplitude, sr):
    short_ms = 0.005
    t = np.linspace(0.0, short_ms, int(sr * short_ms), endpoint=False)
    wave = amplitude * np.sin(2.0 * np.pi * freq * t)

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.plot(t * 1000.0, wave, color="#8B0000", linewidth=2)
    ax.set_title(f"{freq} Hz Waveform (first 5 ms)")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.grid(True)

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    waveform_img = ImageReader(buf)

    general_eq = get_latex_image("s[n] = A\\sin(2\\pi f \\frac{n}{F_s})")
    specific_eq = get_latex_image(
        f"s[n] = {amplitude:.2f}\\sin(2\\pi \\cdot {freq} \\cdot n / {sr})"
    )

    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter
    y = height - 60

    c.setFont("Helvetica-Bold", 18)
    c.drawString(60, y, f"{freq} Hz Healing Solfeggio Tone")
    y -= 35

    c.setFont("Helvetica", 10)
    lead_in = [
        "Research suggests the 528 Hz audio frequency may influence stress-",
        "related biomarkers, including lowering cortisol, the body's primary",
        "stress hormone, and increasing oxytocin, the body’s primary hormone",
        "for bonding, trust, and social connection <3"
    ]
    for line in lead_in:
        c.drawString(60, y, line)
        y -= 14

    y -= 20

    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Equation:")
    y -= 18

    c.setFont("Helvetica", 10)
    c.drawString(60, y, "The general equation for a pure tone sine wave (one single frequency) is defined as:")
    y -= 25

    c.drawImage(general_eq, 60, y - 25, width=150, preserveAspectRatio=True)
    y -= 70

    c.setFont("Helvetica", 10)
    c.drawString(60, y, "Where:")
    y -= 16

    c.drawString(80, y, f"A  = amplitude = {amplitude}")
    y -= 16
    c.drawString(80, y, "f  = frequency in Hz")
    y -= 16
    c.drawString(80, y, "n  = sample index")
    y -= 16
    c.drawString(80, y, f"F_s = sample rate = {sr}")
    y -= 28

    c.setFont("Helvetica", 10)
    c.drawString(60, y, "Worked Example:")
    y -= 20

    c.drawImage(specific_eq, 60, y - 25, width=200, preserveAspectRatio=True)
    y -= 80

    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Waveform Visualization:")
    y -= 12

    c.drawImage(waveform_img, 60, y - 120, width=400, height=120)
    y -= 150

    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "References:")
    y -= 20

    refs = [
        "Fukushima et al. (2018). Effects of 528 Hz on endocrine and mood states.",
        "URL: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5871151/",
        "",
        "Hirokawa et al. (2015). Rat stress markers under 528 Hz.",
        "URL: https://www.jstage.jst.go.jp/article/jvms/77/7/77_150015/_article"
    ]
    for line in refs:
        c.drawString(60, y, line)
        y -= 14

    c.save()


# ============================================================
# 5. CREATE PDF + INLINE PREVIEW + AUTO-DOWNLOAD
# ============================================================
generate_pdf(PDF_FILENAME, CENTER_FREQ_HZ, MASTER_VOLUME, SAMPLE_RATE)

print("Exported PDF:", PDF_FILENAME)
display(IFrame(PDF_FILENAME, width=600, height=800))

if IN_COLAB:
    files.download(PDF_FILENAME)